[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/05_attention.ipynb)

# 🔴 Hard: Softmax Attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

# def scaled_dot_product_attention(Q, K, V):
#     """
#     Compute the scaled dot-product attention.

#     Args:
#         Q: Query matrix of shape (batch_size, seq_len_q, d_k)
#         K: Key matrix of shape (batch_size, seq_len_k, d_k)
#         V: Value matrix of shape (batch_size, seq_len_v, d_v)

#     Returns:
#         Attention output of shape (batch_size, seq_len_q, d_v)
#     """
#     # Step 1: Compute the dot product between Q and K^T
#     # Q: (batch_size, seq_len_q, d_k)
#     # K^T: (batch_size, d_k, seq_len_k)
#     # scores: (batch_size, seq_len_q, seq_len_k)
#     scores = torch.matmul(Q, K.transpose(-2, -1))

#     # Step 2: Scale the scores by sqrt(d_k)
#     d_k = Q.size(-1)  # or K.size(-1), they should be the same
#     scaled_scores = scores / math.sqrt(d_k)

#     # Step 3: Apply softmax to get attention weights
#     # attention_weights: (batch_size, seq_len_q, seq_len_k)
#     attention_weights = torch.softmax(scaled_scores, dim=-1)

#     # Step 4: Multiply the attention weights with V
#     # V: (batch_size, seq_len_v, d_v)
#     # output: (batch_size, seq_len_q, d_v)
#     output = torch.matmul(attention_weights, V)

#     return output


def scaled_dot_product_attention(Q, K, V):
    scores = Q @ K.transpose(-2, -1)  # (batch_size, seq_len_q, d_k) @ (batch_size, d_k, seq_len_k) -> (batch_size, seq_len_q, seq_len_k)
    d_k = Q.size(-1)  # or K.size(-1), they should be the same
    scores_scaled = scores / math.sqrt(d_k)  # Scale the scores
    attention_weights = torch.softmax(scores_scaled, dim=-1)  # Apply softmax to get attention weights
    output = attention_weights @ V  # (batch_size, seq_len_q, seq_len_k) @ (batch_size, seq_len_k, d_v) -> (batch_size, seq_len_q, d_v)
    return output

In [7]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
Cross-attn shape: torch.Size([1, 3, 32])


In [8]:
# ✅ SUBMIT
from torch_judge import check
check("attention")


🧪 Testing: Softmax Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.9ms)
  ✅ [2/4] Numerical correctness (1.4ms)
  ✅ [3/4] Gradient check (1.7ms)
  ✅ [4/4] Cross-attention (seq_q != seq_k) (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (6.1ms total)
  Progress saved. Run status() to see your dashboard.

